# Overview

**[Predicting Heart Disease](https://www.kaggle.com/competitions/playground-series-s6e2/overview)**

Target은 Heart Disease

Learning Type: Binary Classification

Metric: Area under curve

In [1]:
import os
from pathlib import Path

In [2]:
data_path = Path('data')

if not os.path.exists(data_path / 'train.csv'):
    !kaggle competitions download -c playground-series-s6e2
    !unzip playground-series-s6e2.zip -d data
    !rm playground-series-s6e2.zip

In [3]:
import mllabs
mllabs.__version__

'0.3.0'

In [4]:
import polars as pl
import pandas as pd

from sklearn.pipeline import make_pipeline
from mllabs.processor import PolarsLoader, PandasConverter

In [5]:
loader = make_pipeline(
    PolarsLoader(predefined_types={'id': pl.Int64}),
    PandasConverter(index_col='id')
)
train_file = data_path / 'train.csv'
test_file = data_path / 'test.csv'
df_train = loader.fit_transform([train_file])
df_data_spec = loader[0].df_type_.join(
    pd.Series(loader[0].pl_type_, name = 'actual_dtype')
)
df_data_spec

,min,max,na,count,n_unique,dtype,f32,i32,i16,i8,actual_dtype
feature,,,,,,,,,,,
Age,29.0,77.0,0.0,630000.0,42.0,Int64,True,True,True,True,Int8
BP,94.0,200.0,0.0,630000.0,66.0,Int64,True,True,True,False,Int16
Chest pain type,1.0,4.0,0.0,630000.0,4.0,Int64,True,True,True,True,Int8
Cholesterol,126.0,564.0,0.0,630000.0,150.0,Int64,True,True,True,False,Int16
EKG results,0.0,2.0,0.0,630000.0,3.0,Int64,True,True,True,True,Int8
Exercise angina,0.0,1.0,0.0,630000.0,2.0,Int64,True,True,True,True,Int8
FBS over 120,0.0,1.0,0.0,630000.0,2.0,Int64,True,True,True,True,Int8
Heart Disease,NaN,NaN,0.0,630000.0,2.0,String,False,False,False,False,Categorical
Max HR,71.0,202.0,0.0,630000.0,93.0,Int64,True,True,True,False,Int16


# Data Spec.

원본 데이터셋을 기반으로 Competition 용으로 인공적으로 생성한 데이터인데,

이 문제에 접근한 시점에서 원본 데이터가 올라온 페이지가

사라져서 정확한 스펙을 알수가 없다.

데이터에 대한 지식이 없어 GPT를 활용해서 대략의 스펙 조사.

변수의 대략의 스펙을 뽑아 보자.

In [6]:
print(df_data_spec.loc[:, ['min', 'max', 'n_unique', 'actual_dtype']].to_markdown())

| feature                 |   min |      max |   n_unique | actual_dtype   |
|:------------------------|------:|---------:|-----------:|:---------------|
| Age                     |    29 |     77   |         42 | Int8           |
| BP                      |    94 |    200   |         66 | Int16          |
| Chest pain type         |     1 |      4   |          4 | Int8           |
| Cholesterol             |   126 |    564   |        150 | Int16          |
| EKG results             |     0 |      2   |          3 | Int8           |
| Exercise angina         |     0 |      1   |          2 | Int8           |
| FBS over 120            |     0 |      1   |          2 | Int8           |
| Heart Disease           |   nan |    nan   |          2 | Categorical    |
| Max HR                  |    71 |    202   |         93 | Int16          |
| Number of vessels fluro |     0 |      3   |          4 | Int8           |
| ST depression           |     0 |      6.2 |         66 | Float32        |

## Type from GPT

| Feature                  | ML_Type      |
|--------------------------|-------------|
| Age                      | Numeric     |
| BP                       | Numeric     |
| Cholesterol              | Numeric     |
| Max HR                   | Numeric     |
| ST depression            | Numeric     |
| Chest pain type          | Categorical |
| EKG results              | Categorical |
| Exercise angina          | Categorical |
| FBS over 120             | Categorical |
| Sex                      | Categorical |
| Thallium                 | Categorical |
| Number of vessels fluro  | Ordinal     |
| Slope of ST              | Ordinal     |
| Heart Disease            | Target      |
| id                       | Identifier  |

**일단은 위 형식으로 따라가고, 타입 조정의 요소가 있을 때, 변경**

In [7]:
df_test = loader.transform([test_file])

Thallium 이 원소명이라 수치형으로 생각이 됐지만, GPT는 코드라고 응답했고,  범주형이라고 하여, 빈도 조사해 보자.

In [8]:
df_train['Thallium'].value_counts()

Thallium
3    372286
7    246748
6     10966
Name: count, dtype: int64

In [9]:
target = 'Heart Disease'
X_cont = ['Age', 'BP', 'Cholesterol', 'Max HR', 'ST depression']
X_nom = ['Chest pain type', 'EKG results', 'Thallium']
X_ord = ['Number of vessels fluro', 'Slope of ST']
X_bin = ['Exercise angina', 'FBS over 120', 'Sex']
X_all = X_cont + X_nom + X_ord + X_bin
print('Var. no', df_data_spec.shape[0])
print('Cont.', len(X_cont))
print('Nom.', len(X_nom))
print('Ord.', len(X_ord))
print('Bin.', len(X_bin))

Var. no 15
Cont. 5
Nom. 3
Ord. 2
Bin. 3


In [10]:
display(df_train.head(5))
df_train.shape

,Age,Sex,Chest pain type,BP,Cholesterol,FBS over 120,EKG results,Max HR,Exercise angina,ST depression,Slope of ST,Number of vessels fluro,Thallium,Heart Disease
id,,,,,,,,,,,,,,
0,58,1,4,152,239,0,0,158,1,3.6,2,2,7,Presence
1,52,1,1,125,325,0,2,171,0,0.0,1,0,3,Absence
2,56,0,2,160,188,0,2,151,0,0.0,1,0,3,Absence
3,44,0,3,134,229,0,2,150,0,1.0,2,0,3,Absence
4,58,1,4,140,234,0,2,125,1,3.8,2,3,3,Presence


(630000, 14)

In [11]:
display(df_test.head(5))
df_test.shape

,Age,Sex,Chest pain type,BP,Cholesterol,FBS over 120,EKG results,Max HR,Exercise angina,ST depression,Slope of ST,Number of vessels fluro,Thallium
id,,,,,,,,,,,,,
630000,58,1,3,120,288,0,2,145,1,0.8,2,3,3
630001,55,0,2,120,209,0,0,172,0,0.0,1,0,3
630002,54,1,4,120,268,0,0,150,1,0.0,2,3,7
630003,44,0,3,112,177,0,0,168,0,0.9,1,0,3
630004,43,1,1,138,267,0,0,163,0,1.8,2,0,7


(270000, 13)

# 평가셋과 학습셋의 차이 파악

평가셋에 대한 정보가 딱히 없다. Train과 Test의 분포의 차이를 Train과 Test를 분류하는 모델을 만들어 살펴 보고,

피쳐의 특성을 파악할 때, Train과 Test의 분포 차이를 봐야 할 지를 정해 놓고, 속성별 파악을 하자.

In [12]:
df_is_test = pd.concat([
    df_train.drop(columns = [target]).assign(is_test=0),
    df_test.assign(is_test=1)
])
display(df_is_test.head())
df_is_test['is_test'].value_counts()

,Age,Sex,Chest pain type,BP,Cholesterol,FBS over 120,EKG results,Max HR,Exercise angina,ST depression,Slope of ST,Number of vessels fluro,Thallium,is_test
id,,,,,,,,,,,,,,
0,58,1,4,152,239,0,0,158,1,3.6,2,2,7,0
1,52,1,1,125,325,0,2,171,0,0.0,1,0,3,0
2,56,0,2,160,188,0,2,151,0,0.0,1,0,3,0
3,44,0,3,134,229,0,2,150,0,1.0,2,0,3,0
4,58,1,4,140,234,0,2,125,1,3.8,2,3,3,0


is_test
0    630000
1    270000
Name: count, dtype: int64

In [13]:
from mllabs import Experimenter, Connector
from mllabs.collector import MetricCollector, ModelAttrCollector

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import roc_auc_score
from IPython.display import Markdown

In [39]:
import shutil
shutil.rmtree('exp/is_test')

In [40]:
e_is_test = Experimenter.create(
    df_is_test, 'exp/is_test', title = 'Train/Test셋 여부를 구분', sp = StratifiedShuffleSplit(n_splits=1, random_state = 1), splitter_params = {'y': 'is_test'}
)
Markdown(
    e_is_test.desc_spec()
)

📁 Created directory: exp/is_test


## Train/Test셋 여부를 구분

| 항목 | 값 |
|------|-----|
| **Outer Splitter (sp)** | `StratifiedShuffleSplit(n_splits=1, random_state=1)` |
| **Inner Splitter (sp_v)** | None |
| **Splitter Params** | `{y='is_test'}` |
| **Outer Folds** | 1 |
| **Inner Folds** | 1 |

In [41]:
import lightgbm as lgb
e_is_test.add_collector(
    MetricCollector(
        'AUC', Connector(edges = {'y': [(None, 'is_test')]}), slice(-1, None), roc_auc_score, include_train = True
    )
)
e_is_test.add_collector(
    ModelAttrCollector(
        'lgb_feature_importance', Connector(processor = lgb.LGBMClassifier), 'feature_importances'
    )
)
e_is_test.set_grp('clf', role = 'head', edges = {'y': [(None, 'is_test')]}, method = 'predict_proba')
e_is_test.set_node(
    'lgb1', grp='clf', processor=lgb.LGBMClassifier , edges = {'X': [(None, X_all)]}, 
    params={'verbose': -1, 'categorical_features': X_nom}
)

Collect 1/1 (100%) Node 0
Collect 1/1 (100%) Node 0


{'result': 'new',
 'affected_nodes': [],
 'old_obj': None,
 'obj': <mllabs._pipeline.PipelineNode at 0x7f3f0d31c800>}

In [42]:
e_is_test.set_grp('pre', role = 'stage', method = 'transform')

{'result': 'new',
 'grp': <mllabs._pipeline.PipelineGroup at 0x7f3f0c708560>,
 'affected_nodes': []}

In [43]:
from mllabs.processor import CategoricalConverter

In [44]:
e_is_test.set_node(
    'n2c', grp='pre', processor=CategoricalConverter, edges = {'X': [(None, X_nom)]}
)

{'result': 'new',
 'affected_nodes': [],
 'old_obj': None,
 'obj': <mllabs._pipeline.PipelineNode at 0x7f3f0d31cb00>}

In [45]:
e_is_test.set_node(
    'lgb1', grp='clf', processor=lgb.LGBMClassifier , edges = {'X': [(None, X_cont + X_bin), ('n2c', None)]}, 
    params={'verbose': -1, 'categorical_features': ['n2c__' + i for i in X_nom]}, method = 'predict_proba', exist = 'replace'
)

{'result': 'update',
 'affected_nodes': [],
 'old_obj': <mllabs._pipeline.PipelineNode at 0x7f3f0d31c800>,
 'obj': <mllabs._pipeline.PipelineNode at 0x7f3f0c7099a0>}

In [46]:
Markdown(
    e_is_test.desc_pipeline()
)

```mermaid
graph TD

    DataSource([DataSource])
    style DataSource fill:#fff9c4,stroke:#f57c00,stroke-width:3px

    subgraph grp_clf["clf"]
        node_lgb1["lgb1"]
        style node_lgb1 fill:#c8e6c9,stroke:#388e3c,stroke-width:2px
    end
    style grp_clf fill:#e3f2fd,stroke:#1976d2,stroke-width:2px

    subgraph grp_pre["pre"]
        node_n2c["n2c"]
        style node_n2c fill:#c8e6c9,stroke:#388e3c,stroke-width:2px
    end
    style grp_pre fill:#e3f2fd,stroke:#1976d2,stroke-width:2px

    DataSource --> grp_clf
    DataSource --> grp_pre
    grp_pre --> grp_clf
```

In [47]:
Markdown(
    e_is_test.desc_node('lgb1', show_params=True)
)

```mermaid
graph TD

    DataSource([DataSource])
    style DataSource fill:#fff9c4,stroke:#f57c00,stroke-width:3px

    subgraph node_lgb1["clf/lgb1"]
        lgb1_info["<table><tr><td align='left'><b>processor</b></td><td align='left'>LGBMClassifier</td></tr><tr><td align='left'><b>method</b></td><td align='left'>predict_proba</td></tr><tr><td align='left'><b>verbose</b></td><td align='left'>-1</td></tr><tr><td align='left'><b>categorical_features</b></td><td align='left'>['n2c__Chest pain type', 'n2c__EKG re...</td></tr></table>"]
    end
    style node_lgb1 fill:#ffcdd2,stroke:#c62828,stroke-width:3px

    subgraph node_n2c["pre/n2c"]
        n2c_info["<table><tr><td align='left'><b>processor</b></td><td align='left'>CategoricalConverter</td></tr><tr><td align='left'><b>method</b></td><td align='left'>transform</td></tr></table>"]
    end
    style node_n2c fill:#c8e6c9,stroke:#388e3c,stroke-width:2px

    DataSource -->|X,y| node_lgb1
    DataSource --> node_n2c
    node_n2c --> node_lgb1
```

**Path from DataSource to 'clf/lgb1' (2 path(s) found)**

### Edges

| Key | Node | Var |
|-----|------|-----|
| X | Data Source | `['Age', 'BP', 'Cholesterol', 'Max HR', 'ST depression', 'Exercise angina', 'FBS over 120', 'Sex']` |
| X | pre/n2c | * |
| y | Data Source | `is_test` |

In [49]:
e_is_test.build(rebuild=True)

Building 1 node(s)
Build 1/1 (100%) n2c 1/1 (100%)
Build complete: 1 node(s)


In [50]:
e_is_test.exp()

Experimenting 1 node(s)
Exp 1/1 (100%) lgb1 1/1 (100%) 100/100 (100%)
Experimentation complete: 1 node(s)


In [57]:
e_is_test.collectors['AUC'].get_metrics_agg(None)[0]

,valid,train_sub
lgb1,0.503313,0.542511


In [59]:
e_is_test.set_node(
    'lgb2', grp='clf', processor=lgb.LGBMClassifier , edges = {'X': [(None, X_cont + X_bin), ('n2c', None)]}, 
    params={'verbose': -1, 'categorical_features': ['n2c__' + i for i in X_nom], 'n_estimators': 1000}, method = 'predict_proba'
)
e_is_test.exp()

Experimenting 1 node(s)
Exp 1/1 (100%) lgb2 1/1 (100%) 1000/1000 (100%)
Experimentation complete: 1 node(s)


In [60]:
e_is_test.collectors['AUC'].get_metrics_agg(None)[0]

,valid,train_sub
lgb1,0.503313,0.542511
lgb2,0.504157,0.638938
